# APKGC — chạy phần thực nghiệm còn thiếu

Notebook này chạy những thí nghiệm chưa làm được trong lần trước:

1. **Ba seed còn lại** (1234, 2024, 3407) trên MKG-W — để có độ lệch chuẩn tin cậy
2. **Cấu hình đúng như bài báo** (batch 2048, neg 64) — tách nguyên nhân khoảng lệch 0.011
3. **Ablation tự chạy** — bỏ noise sampling, đổi cách hợp nhất

Chọn phần cần chạy ở mục 5; Kaggle giới hạn 12 giờ mỗi phiên nên không nên bật hết cùng lúc.

---

**Nguyên tắc:** siêu tham số đọc trực tiếp từ `scripts/run_base.sh` của tác giả. Mỗi thí nghiệm chỉ thay đổi đúng một biến, và biến đó được ghi vào tệp kết quả.

## 1. Kiểm tra GPU

In [ ]:
import torch, subprocess, sys

print(subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
print("torch:", torch.__version__)

NEED_DOWNGRADE = False
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name()}  sm_{cap[0]}{cap[1]}")
    try:
        _ = (torch.randn(8,8,device="cuda") @ torch.randn(8,8,device="cuda")).sum().item()
        print("\nGPU chay duoc.")
    except Exception as e:
        print(f"\nGPU khong chay duoc: {type(e).__name__}")
        NEED_DOWNGRADE = True
else:
    print("Khong thay GPU.")


### Xử lý khi GPU không tương thích

In [ ]:
# Kaggle doi khi cap Tesla P100 (sm_60) trong khi ban torch cai san chi ho tro
# tu sm_70. Ha ve ban co kernel cho sm_60. Khong can khoi dong lai kernel vi
# phan huan luyen chay qua subprocess.
if NEED_DOWNGRADE:
    print("Dang cai torch 2.4.1+cu121 ...")
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -2
    import subprocess, sys
    r = subprocess.run([sys.executable,"-c",
        "import torch;a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__);print('matmul OK',float((a@a).sum()))"],
        capture_output=True, text=True)
    print(r.stdout or r.stderr[-400:])
else:
    print("Khong can thay doi torch.")


## 2. Tải mã nguồn và chuẩn bị

In [ ]:
%cd /kaggle/working
!rm -rf APKGC _openke
!git clone -q https://github.com/HubuKG/APKGC.git
!ls APKGC

In [ ]:
import os, glob, subprocess, sys

A = "/kaggle/working/APKGC"

# 1. Bo sung ma nguon C++ con thieu.
#    Repo chi commit Base.so bien dich san cho Linux x86-64; thu muc
#    mmkgc/base/ chua Base.cpp khong duoc commit nen khong bien dich lai duoc.
#    mmkgc/ la ban sua doi cua OpenKE (THUNLP) nen lay base/ tu do.
if not os.path.exists(f"{A}/mmkgc/base/Base.cpp"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/thunlp/OpenKE.git",
                    "/kaggle/working/_openke"], check=True)
    subprocess.run(["cp","-r","/kaggle/working/_openke/openke/base",
                    f"{A}/mmkgc/base"], check=True)
    print("Da bo sung mmkgc/base/ tu OpenKE")

# 2. Module chon thiet bi
DEVICE_SRC = "\n".join([
    "import os", "import torch",
    '_f = os.environ.get("MMKGC_DEVICE", "").strip().lower()',
    "if _f:", "    DEVICE = torch.device(_f)",
    "elif torch.cuda.is_available():", '    DEVICE = torch.device("cuda")',
    "else:", '    DEVICE = torch.device("cpu")',
    "def device_name():", "    return str(DEVICE)", "",
])
open(f"{A}/mmkgc/device.py","w").write(DEVICE_SRC)

# 3. Thay .cuda() cung
total = 0
for p in glob.glob(f"{A}/**/*.py", recursive=True):
    if p.endswith("device.py"):
        continue
    s = open(p, encoding="utf-8").read()
    if ".cuda()" not in s:
        continue
    n = s.count(".cuda()")
    rel = os.path.relpath(p, A)
    if rel.startswith("mmkgc/"):
        depth = os.path.relpath(p, f"{A}/mmkgc").count(os.sep)
        imp = "from " + "."*(depth+1) + "device import DEVICE"
    else:
        imp = "from mmkgc.device import DEVICE"
    if "device import DEVICE" not in s:
        lines = s.split("\n")
        last = max((i for i,l in enumerate(lines)
                    if l.startswith(("import ","from "))), default=0)
        lines.insert(last+1, imp)
        s = "\n".join(lines)
    s = s.replace(".cuda()", ".to(DEVICE)")
    open(p,"w",encoding="utf-8").write(s)
    total += n
print(f"Da thay {total} cho .cuda()")

# 4. Sua cac loi khien khong chay duoc
p = f"{A}/mmkgc/config/Trainer.py"
s = open(p, encoding="utf-8").read()
s = s.replace("data_loader=self.test_dataloader, use_gpu=True)",
              "data_loader=self.test_dataloader, use_gpu=(DEVICE.type != 'cpu'))")
if "self.best_model_wts = None" not in s:
    s = s.replace("        self.Loss_log = Loss_log(early_stop=args.early_stop)",
                  "        self.Loss_log = Loss_log(early_stop=args.early_stop)\n"
                  "        self.best_model_wts = None")
open(p,"w",encoding="utf-8").write(s)

p = f"{A}/args.py"
s = open(p, encoding="utf-8").read()
if "save_steps" not in s:
    s = s.replace("    arg.add_argument('-num_proj', type=int, default=1)",
                  "    arg.add_argument('-num_proj', type=int, default=1)\n"
                  "    arg.add_argument('-save_steps', type=int, default=100)")
open(p,"w",encoding="utf-8").write(s)

p = f"{A}/run_KGC_train.py"
s = open(p, encoding="utf-8").read()
s = s.replace("    torch.cuda.manual_seed_all(args.seed)",
              "    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)")
s = s.replace("    data_root = osp.abspath(osp.join(this_dir, '..', '..', 'data', ''))",
              "    data_root = osp.abspath(osp.join(this_dir, 'output'))")
s = s.replace("        save_steps=100,", "        save_steps=args.save_steps,")
s = s.replace("use_gpu=True", "use_gpu=(DEVICE.type != 'cpu')")
if "from mmkgc.device import DEVICE" not in s:
    s = s.replace("from args import get_args",
                  "from args import get_args\nfrom mmkgc.device import DEVICE, device_name")
OLD = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if not osp.exists(save_dir):",
    '        torch.save(trainer.best_model_wts, f"{args.save}-MRR{trainer.Loss_log.get_acc()}")',
    "", "    kge_score.load_checkpoint(save_dir)"])
NEW = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if trainer.best_model_wts is not None:",
    "        if not osp.exists(save_dir):",
    "            torch.save(trainer.best_model_wts, save_dir)",
    "        kge_score.load_checkpoint(save_dir)"])
if OLD in s:
    s = s.replace(OLD, NEW)
open(p,"w",encoding="utf-8").write(s)
print("Da sua Trainer/args/entrypoint")


In [ ]:
%cd /kaggle/working/APKGC/mmkgc
!bash make.sh 2>&1 | tail -2
!file release/Base.so
%cd /kaggle/working

In [ ]:
!pip install -q transformers easydict unidecode tensorboard scikit-learn scipy 2>&1 | tail -1
print('deps OK')

## 3. Tải bộ nhúng

In [ ]:
!pip install -q gdown
%cd /kaggle/working/APKGC
!gdown --folder 1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB -O ./emb_dl -q

import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
for f in pathlib.Path('emb_dl').rglob('*'):
    if f.suffix in ('.pth','.json'):
        dst = pathlib.Path('embeddings')/f.name
        if not dst.exists(): shutil.move(str(f), str(dst))
!ls -la embeddings/

## 4. Bộ khung chạy thí nghiệm

In [ ]:
import subprocess, sys, os, re, json, time, statistics

A   = "/kaggle/working/APKGC"
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

LOGGER_LINE = re.compile(
    r"mrr:\s*([\d.]+).*?mr:\s*([\d.]+).*?hit10:\s*([\d.]+)"
    r".*?hit3:\s*([\d.]+).*?hit1:\s*([\d.]+)")
SHELL_VAR = re.compile(r"^\s*([A-Z_][A-Z0-9_]*)=(\S+)\s*$", re.MULTILINE)


def read_author_config():
    """Doc sieu tham so tu chinh script tac gia goi trong run.sh."""
    cfg = dict(SHELL_VAR.findall(open(f"{A}/scripts/run_base.sh").read()))
    for k in ["EMB_DIM","NUM_BATCH","MARGIN","LR","NEG_NUM","EPOCH"]:
        assert k in cfg, f"thieu {k}"
    return cfg


def parse_metrics(text):
    hits = LOGGER_LINE.findall(text)
    if not hits:
        return None
    m = hits[-1]
    return dict(mrr=float(m[0]), mr=float(m[1]), hit10=float(m[2]),
                hit3=float(m[3]), hit1=float(m[4]))


# Theo dung run.sh goc:
#   dong 1: DB15K  num_proj=2  Mformer_hd_graph
#   dong 2: MKG-W  num_proj=1  Mformer_hd_mean
DATASET_ARGS = {
    "DB15K": dict(num_proj=2, joint_way="Mformer_hd_graph"),
    "MKG-W": dict(num_proj=1, joint_way="Mformer_hd_mean"),
    "MKG-Y": dict(num_proj=1, joint_way="Mformer_hd_mean"),
}


def run_exp(tag, dataset, cfg, seed=42, epoch=None, save_steps=250,
            add_noise=None, joint_way=None, batch=None, neg=None,
            noise_ratio=0.2, mask_ratio=0.7):
    """Chay mot thi nghiem. Moi tham so de None thi lay tu cau hinh tac gia."""
    da = DATASET_ARGS[dataset]
    n_epoch = epoch or int(cfg["EPOCH"])
    os.makedirs(f"{A}/output/mmkgc/checkpoint", exist_ok=True)

    cmd = [sys.executable, "-u", "run_KGC_train.py",
           f"-dataset={dataset}",
           f"-num_proj={da['num_proj']}",
           "-use_intermediate=0",
           f"-joint_way={joint_way or da['joint_way']}",
           f"-batch_size={batch or cfg['NUM_BATCH']}",
           f"-margin={cfg['MARGIN']}",
           f"-epoch={n_epoch}",
           f"-dim={cfg['EMB_DIM']}",
           f"-neg_num={neg or cfg['NEG_NUM']}",
           f"-learning_rate={cfg['LR']}",
           f"-noise_update={cfg.get('NU','epoch')}",
           f"-noise_ratio={noise_ratio}", f"-mask_ratio={mask_ratio}",
           f"-use_pool={cfg.get('POOL',1)}",
           f"-add_noise={cfg.get('NOISE',1) if add_noise is None else add_noise}",
           "-num_hidden_layers=1", "-num_attention_heads=2",
           f"-seed={seed}", f"-save={tag}",
           f"-save_steps={save_steps}", f"-exp_id={tag}"]

    print(f"  [{tag}] bat dau ...", flush=True)
    t0 = time.time()
    evals = []
    log_path = f"{OUT}/{tag}.log"

    with open(log_path, "w", buffering=1) as lf:
        proc = subprocess.Popen(cmd, cwd=A, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            lf.write(line)
            m = LOGGER_LINE.search(line)
            if m:
                ev = dict(mrr=float(m.group(1)), mr=float(m.group(2)),
                          hit10=float(m.group(3)), hit3=float(m.group(4)),
                          hit1=float(m.group(5)),
                          elapsed=round(time.time()-t0,1))
                evals.append(ev)
                print(f"    eval {len(evals):>2}: MRR={ev['mrr']:.4f} "
                      f"({ev['elapsed']/60:.0f} phut)", flush=True)
                # ghi ngay sau moi lan eval, phong kernel bi cat
                json.dump(dict(tag=tag, dataset=dataset, evals=evals,
                               best=max(evals, key=lambda e: e["mrr"])),
                          open(f"{OUT}/partial_{tag}.json","w"), indent=2)
        proc.wait()

    dt = time.time() - t0
    if not evals:
        print(f"    that bai sau {dt/60:.1f} phut — xem {log_path}")
        return None

    best = max(evals, key=lambda e: e["mrr"])
    rec = dict(best)
    rec.update(tag=tag, dataset=dataset, seed=seed, epoch=n_epoch,
               save_steps=save_steps, seconds=round(dt,1), n_evals=len(evals),
               cmd=" ".join(cmd[2:]))
    json.dump(rec, open(f"{OUT}/{tag}.json","w"), indent=2)
    print(f"    xong: MRR={rec['mrr']:.4f}  H@1={rec['hit1']:.4f}  "
          f"({dt/60:.0f} phut, {len(evals)} lan eval)")
    return rec


In [ ]:
cfg = read_author_config()
print('Cau hinh tu scripts/run_base.sh:')
for k,v in cfg.items(): print(f'  {k:<12} = {v}')
print()
print('Bai bao (tr.15010) ghi: dim=128, batch=2048, neg=64, lr=1e-4')
print('=> batch va neg trong script chi bang mot nua gia tri cong bo.')

## 5. Chọn thí nghiệm cần chạy

Bật `True` cho phần muốn chạy. Ước lượng thời gian dựa trên lần chạy trước (khoảng 2 giờ 40 phút mỗi lần với early stopping).

In [ ]:
RUN_SEEDS        = True    # 3 seed con lai       ~8 gio
RUN_PAPER_CONFIG = False   # batch 2048 / neg 64  ~3 gio
RUN_ABLATION     = False   # bo noise, doi fusion ~6 gio

EVAL_EVERY = 100   # danh gia moi 250 epoch (tac gia dung 100)
EPOCH_CAP  = 8000  # gioi han epoch cho vua phien Kaggle

print('Se chay:')
if RUN_SEEDS:        print('  - 3 seed: 1234, 2024, 3407')
if RUN_PAPER_CONFIG: print('  - cau hinh bai bao: batch 2048, neg 64')
if RUN_ABLATION:     print('  - ablation: bo noise / doi fusion')

### 5.1 Ba seed còn lại

Cùng cấu hình với hai seed đã chạy (batch 1024, neg 32) để cộng vào cùng một mẫu thống kê.

In [ ]:
results = {}

if RUN_SEEDS:
    for sd in [42]:
        r = run_exp(f'seed{sd}', 'MKG-W', cfg, seed=sd,
                    epoch=EPOCH_CAP, save_steps=EVAL_EVERY)
        if r: results[f'seed{sd}'] = r

### 5.2 Cấu hình đúng như bài báo

Bài báo ghi batch 2048 và 64 mẫu âm; script lại đặt 1024 và 32. Thí nghiệm này kiểm tra xem khoảng lệch 0.011 quan sát được có đến từ khác biệt này không.

In [ ]:
if RUN_PAPER_CONFIG:
    r = run_exp('paper_config', 'MKG-W', cfg, seed=42,
                epoch=EPOCH_CAP, save_steps=EVAL_EVERY,
                batch=2048, neg=64)
    if r: results['paper_config'] = r

### 5.3 Ablation tự chạy

Bảng 4 của bài báo báo cáo bỏ noise sampling làm MRR giảm từ 0.374 xuống 0.358. Ở đây nhóm tự đo lại phần này.

In [ ]:
if RUN_ABLATION:
    r = run_exp('no_noise', 'MKG-W', cfg, seed=42,
                epoch=EPOCH_CAP, save_steps=EVAL_EVERY, add_noise=0)
    if r: results['no_noise'] = r

    r = run_exp('joint_graph', 'MKG-W', cfg, seed=42,
                epoch=EPOCH_CAP, save_steps=EVAL_EVERY,
                joint_way='Mformer_hd_graph')
    if r: results['joint_graph'] = r

## 6. Tổng hợp

In [ ]:
import json, glob, statistics

# Gop ca ket qua lan chay truoc de tinh do lech chuan tren nhieu seed hon
PREVIOUS = [
    dict(tag='seed42', mrr=0.3641117811203003, hit1=0.295624703168869,
         hit3=0.3954141438007355, hit10=0.48397284746170044, seed=42),
    dict(tag='seed7',  mrr=0.3615906536579132, hit1=0.29422086477279663,
         hit3=0.3917875587940216, hit10=0.4852597117424011, seed=7),
]

seed_runs = list(PREVIOUS)
for f in sorted(glob.glob('/kaggle/working/results/seed*.json')):
    if 'partial' in f: continue
    seed_runs.append(json.load(open(f)))

print(f"{'nguon':>10} {'seed':>6} {'MRR':>9} {'H@1':>9} {'H@3':>9} {'H@10':>9}")
for r in sorted(seed_runs, key=lambda x: x['seed']):
    src = 'lan truoc' if r['tag'] in ('seed42','seed7') else 'lan nay'
    print(f"{src:>10} {r['seed']:>6} {r['mrr']:>9.4f} {r['hit1']:>9.4f} "
          f"{r['hit3']:>9.4f} {r['hit10']:>9.4f}")

if len(seed_runs) > 1:
    print()
    for k in ('mrr','hit1','hit3','hit10'):
        v = [r[k] for r in seed_runs]
        print(f"{k.upper():>6}: {statistics.mean(v):.4f} +/- {statistics.stdev(v):.4f}"
              f"   bien do {max(v)-min(v):.4f}")

In [ ]:
PUB = dict(mrr=0.374, hit1=0.306, hit3=0.404, hit10=0.501)

if len(seed_runs) > 1:
    std  = statistics.stdev([r['mrr'] for r in seed_runs])
    mean = statistics.mean([r['mrr'] for r in seed_runs])
    print('='*62)
    print('DIEN GIAI')
    print('='*62)
    print(f'So cong bo (Bang 2)   : {PUB["mrr"]:.4f}')
    print(f'So do lai ({len(seed_runs)} seed) : {mean:.4f} +/- {std:.4f}')
    print(f'Chenh lech            : {mean-PUB["mrr"]:+.4f}')
    print()
    for lbl, gap in [('APKGC vs NativE (MKG-W)', 0.008),
                     ('Dong gop attention penalty', 0.014),
                     ('Dong gop noise sampling', 0.016)]:
        v = 'nam trong nhieu seed' if std >= gap else 'vuot nhieu seed'
        print(f'  {lbl:<30} {gap:.4f}  ->  {v}')

if 'paper_config' in results:
    pc = results['paper_config']
    base = statistics.mean([r['mrr'] for r in seed_runs])
    print()
    print('CAU HINH BAI BAO (batch 2048, neg 64)')
    print(f'  MRR = {pc["mrr"]:.4f}')
    print(f'  so voi cau hinh script : {pc["mrr"]-base:+.4f}')
    print(f'  so voi so cong bo      : {pc["mrr"]-PUB["mrr"]:+.4f}')

for tag, lbl, ref in [('no_noise','Bo noise sampling',0.358),
                      ('joint_graph','Doi fusion sang graph',None)]:
    if tag in results:
        r = results[tag]
        print()
        print(f'{lbl}: MRR = {r["mrr"]:.4f}')
        if ref: print(f'  bai bao bao cao: {ref:.3f}')

## 7. Kết quả đã lưu

In [ ]:
!ls -la /kaggle/working/results/
import json, glob
for f in sorted(glob.glob('/kaggle/working/results/*.json')):
    if 'partial' in f: continue
    d = json.load(open(f))
    print(f"\n{f.split('/')[-1]}")
    print(f"  MRR={d.get('mrr',0):.4f}  {d.get('n_evals','?')} lan eval  "
          f"{d.get('seconds',0)/60:.0f} phut")
    print(f"  {d.get('cmd','')[:110]}")

---

### Phạm vi kết luận

Các số đo được nói về hành vi của mã nguồn công bố trên phần cứng và phiên bản thư viện cụ thể ở đây. Chúng không cho phép suy ra tác giả đã chạy bao nhiêu lần hay thử bao nhiêu cấu hình — kho mã chỉ phản ánh những gì được công bố.